In [1]:

import fitz  # PyMuPDF
import pytesseract
from PIL import Image
import io
import openai
import chromadb
import os
import openai
from dotenv import load_dotenv
from PyPDF2 import PdfReader
import tiktoken
from chromadb.config import Settings
from openai import OpenAI


In [2]:

load_dotenv()

True

In [3]:
PDF_FOLDER = "./Doc"  # Path to your resume PDF
CHROMA_DIR = "./doc_store"
EMBED_MODEL = "text-embedding-3-large"  # OpenAI embedding model
MAX_TOKENS = 500  # slightly below 8192 for safety


In [4]:
CHROMA_DIR = "./chroma_db"
os.makedirs(CHROMA_DIR, exist_ok=True)

In [5]:
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

In [16]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [7]:
collection = chroma_client.get_or_create_collection("Document")

In [8]:
def read_pdfs_from_folder(folder_path):
    pdf_texts = []
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(folder_path, filename)
            pdf_text = extract_text_f1rom_pdf(pdf_path)
            pdf_texts.append((filename, pdf_text))
    return pdf_texts

In [9]:
def read_pdfs_from_folder(folder_path):
    pdf_texts = []
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(folder_path, filename)
            pdf_text = extract_text_from_pdf(pdf_path)
            pdf_texts.append((filename, pdf_text))
    return pdf_texts

In [10]:
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() or ""
    return text.strip()

In [11]:
# Function to count tokens
def num_tokens_from_string(string, encoding_name="cl100k_base"):
    encoding = tiktoken.get_encoding(encoding_name)
    return len(encoding.encode(string))

In [12]:
def chunk_text(text, chunk_size=1000, overlap=100):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap

    return chunks

In [13]:
def get_openai_embedding(text):
    return client.embeddings.create(
        model="text-embedding-3-large",
        input=text
    ).data[0].embedding

In [17]:
pdf_files_data = read_pdfs_from_folder(PDF_FOLDER)

for filename, text in pdf_files_data:
    print(f"{filename} → extracted {len(text)} characters")

    if not text.strip():
        print(f"⚠ Skipping {filename} — No text found.")
        continue

    chunks = chunk_text(text)

    for i, chunk in enumerate(chunks):
        embedding = get_openai_embedding(chunk)
        print(f"✅ {filename} | Chunk {i} | Embedding length: {len(embedding)}")

Data Science Interview Q&A.pdf → extracted 476347 characters
✅ Data Science Interview Q&A.pdf | Chunk 0 | Embedding length: 3072
✅ Data Science Interview Q&A.pdf | Chunk 1 | Embedding length: 3072
✅ Data Science Interview Q&A.pdf | Chunk 2 | Embedding length: 3072
✅ Data Science Interview Q&A.pdf | Chunk 3 | Embedding length: 3072
✅ Data Science Interview Q&A.pdf | Chunk 4 | Embedding length: 3072
✅ Data Science Interview Q&A.pdf | Chunk 5 | Embedding length: 3072
✅ Data Science Interview Q&A.pdf | Chunk 6 | Embedding length: 3072
✅ Data Science Interview Q&A.pdf | Chunk 7 | Embedding length: 3072
✅ Data Science Interview Q&A.pdf | Chunk 8 | Embedding length: 3072
✅ Data Science Interview Q&A.pdf | Chunk 9 | Embedding length: 3072
✅ Data Science Interview Q&A.pdf | Chunk 10 | Embedding length: 3072
✅ Data Science Interview Q&A.pdf | Chunk 11 | Embedding length: 3072
✅ Data Science Interview Q&A.pdf | Chunk 12 | Embedding length: 3072
✅ Data Science Interview Q&A.pdf | Chunk 13 | Embedd

In [29]:
for idx, (filename, text) in enumerate(pdf_files_data):

    if not text.strip():
        continue

    chunks = chunk_text(text)

    for i, chunk in enumerate(chunks):
        embedding = get_openai_embedding(chunk)

        doc_id = f"{filename}_chunk_{i}"  # ✅ UNIQUE

        collection.add(
            embeddings=[embedding],
            documents=[chunk],
            metadatas=[{"filename": filename}],
            ids=[doc_id]
        )

        print(f"Stored {doc_id}")

Stored Data Science Interview Q&A.pdf_chunk_0
Stored Data Science Interview Q&A.pdf_chunk_1
Stored Data Science Interview Q&A.pdf_chunk_2
Stored Data Science Interview Q&A.pdf_chunk_3
Stored Data Science Interview Q&A.pdf_chunk_4
Stored Data Science Interview Q&A.pdf_chunk_5
Stored Data Science Interview Q&A.pdf_chunk_6
Stored Data Science Interview Q&A.pdf_chunk_7
Stored Data Science Interview Q&A.pdf_chunk_8
Stored Data Science Interview Q&A.pdf_chunk_9
Stored Data Science Interview Q&A.pdf_chunk_10
Stored Data Science Interview Q&A.pdf_chunk_11
Stored Data Science Interview Q&A.pdf_chunk_12
Stored Data Science Interview Q&A.pdf_chunk_13
Stored Data Science Interview Q&A.pdf_chunk_14
Stored Data Science Interview Q&A.pdf_chunk_15
Stored Data Science Interview Q&A.pdf_chunk_16
Stored Data Science Interview Q&A.pdf_chunk_17
Stored Data Science Interview Q&A.pdf_chunk_18
Stored Data Science Interview Q&A.pdf_chunk_19
Stored Data Science Interview Q&A.pdf_chunk_20
Stored Data Science Int

In [30]:
print("Final count:", collection.count())

Final count: 530


In [31]:
def query_rag(question):
    query_embedding = get_openai_embedding(question)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3  # top 3 relevant chunks
    )

    return results

In [32]:
def get_context(results):
    documents = results['documents'][0]
    return "\n\n".join(documents)

In [33]:
def generate_answer(question):
    results = query_rag(question)
    context = get_context(results)

    prompt = f"""
    Answer the question based ONLY on the context below:

    Context:
    {context}

    Question:
    {question}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",  # or your model
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [35]:
generate_answer('What is Data Science')

'Data Science is a blend of various tools, algorithms, and machine learning principles with the goal to discover hidden patterns from raw data. It differs from traditional statistics in that statisticians typically work a posteriori, explaining results and designing plans, while data scientists use historical data to make predictions.'

In [36]:
from telegram import Update
from telegram.ext import ApplicationBuilder, CommandHandler, ContextTypes

In [51]:
TOKEN = "AAG30Jru8HtuQjMqknWbkl1-cp04rny2UlI"

In [52]:

app = ApplicationBuilder().token(TOKEN).build()
print("✅ Bot initialized successfully")

✅ Bot initialized successfully


In [42]:
async def ask_command(update: Update, context: ContextTypes.DEFAULT_TYPE):
    if not context.args:
        await update.message.reply_text("❌ Usage: /ask <your question>")
        return

    question = " ".join(context.args)

    await update.message.reply_text("🤔 Thinking...")

    try:
        answer = generate_answer(question)
        await update.message.reply_text(answer)

    except Exception as e:
        await update.message.reply_text(f"❌ Error: {str(e)}")

In [53]:
def main():
    app = ApplicationBuilder().token(TOKEN).build()

    app.add_handler(CommandHandler("ask", ask_command))

    print("🤖 Bot is running...")
    app.run_polling()


if __name__ == "__main__":
    main()

🤖 Bot is running...


RuntimeError: Cannot close a running event loop